# Weight Histograms & Gradient Tracking with PyTorch + TensorBoard

Training a CNN on **CIFAR-10** and using TensorBoard to visualise weight distributions and gradient flow across every layer.

Running **two experiments back-to-back**:

| Run | Setup |  |
|-----|-------|----------------------|
| `run_no_bn` | CNN without Batch Normalization |
| `run_with_bn` | CNN with Batch Normalization |


## 0. Imports & Setup

In [1]:
# Install if not installed

# !pip install -r requirements.txt
# !pip install torch torchvision tensorboard

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import os
import shutil

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.10.0+cu128


## 1. Loading CIFAR-10

Dwnloading and preparing CIFAR-10 — 60,000 colour images across 10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck).

Applying standard normalisation using the CIFAR-10 channel means and standard deviations, and adding light random flipping for training augmentation.

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

# CIFAR-10 channel means and stds (precomputed)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
val_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_val
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=128, shuffle=False, num_workers=2)

CLASSES = train_dataset.classes
print(f'Classes: {CLASSES}')
print(f'Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}')

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Training batches: 391, Validation batches: 79


## 2. Defining the Models

I'm defining two versions of the same CNN architecture. The only difference is whether batch normalisation layers are included.

The network has:
- 3 convolutional blocks (each with Conv → [optional BN] → ReLU → MaxPool)
- 2 fully connected layers

Three conv blocks gives us enough depth that gradient flow differences between early and late layers become clearly visible in TensorBoard.

In [5]:
class CIFAR10_CNN(nn.Module):
    def __init__(self, use_batchnorm=True):
        super().__init__()
        self.use_batchnorm = use_batchnorm

        # --- Block 1 ---
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()

        # --- Block 2 ---
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64) if use_batchnorm else nn.Identity()

        # --- Block 3 ---
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128) if use_batchnorm else nn.Identity()

        self.pool    = nn.MaxPool2d(2, 2)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        # After 3 max-pools: 32x32 → 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))   # 32x32 → 16x16
        x = self.pool(self.relu(self.bn2(self.conv2(x))))   # 16x16 → 8x8
        x = self.pool(self.relu(self.bn3(self.conv3(x))))   # 8x8   → 4x4
        x = x.view(x.size(0), -1)                           # flatten
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


# Sanity check
dummy = torch.zeros(4, 3, 32, 32)
m = CIFAR10_CNN(use_batchnorm=True)
out = m(dummy)
print(f'Output shape: {out.shape}')  # Should be [4, 10]

total_params = sum(p.numel() for p in m.parameters())
print(f'Total parameters: {total_params:,}')

Output shape: torch.Size([4, 10])
Total parameters: 620,810


## 3. The TensorBoard Logger

Creating a custom logger class that wraps `SummaryWriter` and handles all our TensorBoard logging in one place. This keeps our training loop clean.

Each epoch it is logging:
- Loss and accuracy scalars for train and validation
- Weight and bias **histograms** for every named layer
- **Gradient norms** (L2 norm of `.grad`) for every layer — as scalars, so they show up as a line chart over time

In [6]:
class TBLogger:
    def __init__(self, log_dir):
        if os.path.exists(log_dir):
            shutil.rmtree(log_dir)
        self.writer = SummaryWriter(log_dir)
        print(f'Logging to: {log_dir}')

    def log_scalars(self, tag_value_dict, step):
        """Log multiple scalars at once."""
        for tag, value in tag_value_dict.items():
            self.writer.add_scalar(tag, value, step)

    def log_weights(self, model, step):
        """Log weight and bias histograms for all layers."""
        for name, param in model.named_parameters():
            if param.requires_grad and param.data is not None:
                # Weight histogram
                self.writer.add_histogram(
                    f'weights/{name}', param.data.cpu(), step
                )

    def log_gradients(self, model, step):
        """Log gradient L2 norms per layer as scalars."""
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                grad_norm = param.grad.norm(2).item()
                self.writer.add_scalar(
                    f'grad_norms/{name}', grad_norm, step
                )

    def close(self):
        self.writer.flush()
        self.writer.close()
        print('TensorBoard writer closed.')

## 4. Training Loop

Defining one training function that works for both model variants. I'm logging gradients **immediately after** `loss.backward()` — at that moment the `.grad` tensors on each parameter are freshly populated and ready to read.

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return total_loss / total, correct / total


def run_experiment(run_name, use_batchnorm, epochs=10):
    print(f'\n{"="*55}')
    print(f'  Starting experiment: {run_name}')
    print(f'  Batch Norm: {use_batchnorm} | Epochs: {epochs}')
    print(f'{"="*55}')

    model     = CIFAR10_CNN(use_batchnorm=use_batchnorm).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    logger    = TBLogger(f'runs/{run_name}')

    for epoch in range(1, epochs + 1):
        # --- Train ---
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE
        )

        # --- Evaluate ---
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        # --- Log scalars ---
        logger.log_scalars({
            'Loss/train': train_loss,
            'Loss/val':   val_loss,
            'Accuracy/train': train_acc,
            'Accuracy/val':   val_acc,
            'LearningRate':   scheduler.get_last_lr()[0],
        }, step=epoch)

        # --- Log weight histograms ---
        logger.log_weights(model, step=epoch)

        # --- Log gradient norms (one more forward+backward pass) ---
        # We run a single mini-batch just to populate .grad tensors for logging
        model.train()
        sample_images, sample_labels = next(iter(train_loader))
        sample_images = sample_images.to(DEVICE)
        sample_labels = sample_labels.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(sample_images), sample_labels).backward()
        logger.log_gradients(model, step=epoch)

        scheduler.step()

        print(
            f'Epoch {epoch:02d}/{epochs} | '
            f'Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | '
            f'Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}'
        )

    logger.close()
    return model

## 5. Running the Experiments

Running both experiments sequentially. Each one is writing its TensorBoard logs to a separate subfolder under `runs/`

In [9]:
EPOCHS = 20

model_no_bn   = run_experiment('run_no_bn',   use_batchnorm=False, epochs=EPOCHS)
model_with_bn = run_experiment('run_with_bn', use_batchnorm=True,  epochs=EPOCHS)


  Starting experiment: run_no_bn
  Batch Norm: False | Epochs: 20
Logging to: runs/run_no_bn
Epoch 01/20 | Train Loss: 1.4801 Acc: 0.463 | Val Loss: 1.1746 Acc: 0.585
Epoch 02/20 | Train Loss: 1.0875 Acc: 0.614 | Val Loss: 0.9405 Acc: 0.670
Epoch 03/20 | Train Loss: 0.9073 Acc: 0.681 | Val Loss: 0.8661 Acc: 0.700
Epoch 04/20 | Train Loss: 0.7977 Acc: 0.722 | Val Loss: 0.7697 Acc: 0.732
Epoch 05/20 | Train Loss: 0.7272 Acc: 0.747 | Val Loss: 0.7161 Acc: 0.752
Epoch 06/20 | Train Loss: 0.6191 Acc: 0.787 | Val Loss: 0.6638 Acc: 0.768
Epoch 07/20 | Train Loss: 0.5794 Acc: 0.799 | Val Loss: 0.6539 Acc: 0.774
Epoch 08/20 | Train Loss: 0.5524 Acc: 0.807 | Val Loss: 0.6318 Acc: 0.780
Epoch 09/20 | Train Loss: 0.5215 Acc: 0.818 | Val Loss: 0.6252 Acc: 0.787
Epoch 10/20 | Train Loss: 0.4957 Acc: 0.827 | Val Loss: 0.6237 Acc: 0.785
Epoch 11/20 | Train Loss: 0.4434 Acc: 0.845 | Val Loss: 0.6166 Acc: 0.789
Epoch 12/20 | Train Loss: 0.4244 Acc: 0.853 | Val Loss: 0.5962 Acc: 0.797
Epoch 13/20 | Trai

## 6. Launch TensorBoard

Launching TensorBoard pointing at the `runs/` directory. Both experiments will appear and can be toggled on/off.

In [10]:
%load_ext tensorboard
%tensorboard --logdir runs

If TensorBoard doesn't appear inline, open a terminal and run:
```bash
tensorboard --logdir runs
```
Then visit: **http://localhost:6006**